# K6 Lite Diffusers Weights Media

Приватная копия части `media` закреплённого Diffusers snapshot [kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers](https://huggingface.co/kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers/tree/4c487e8b4aeeb9059e5a77905aa2ddb59ec28f81) @ `4c487e8b`: DiT, video VAE, audio VAE, vocoder, CLIP, tokenizers, scheduler (~9.4 GiB). Kaggle сохраняет не больше 20 GB Output, поэтому snapshot (24,9 GiB) разделён на части `media` и `text`; вторая часть — notebook «K6 Lite Diffusers Weights Text».

Запуск: **Save Version → Save & Run All**, GPU не нужен, Internet On. Затем в notebook 16 добавьте обе части через **Add Input → Your Work → Notebook Output**: ячейка конфигурации найдёт их сама и подставит `LOCAL_MODEL_DIR`, повторные загрузки весов в каждой сессии исчезнут.

Лицензии: Kandinsky 6 — MIT, Qwen2.5-VL и CLIP — их собственные лицензии в исходных репозиториях. Notebook и Output приватные.

Учебные материалы отдельно: [MIT 6.S184](https://diffusion.csail.mit.edu/2026/index.html) · [конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Diffusers runner](https://github.com/vladislav-vasilenko/generative-ai-research/tree/main/kandinsky-6/generation).


In [ ]:
PART='media'
MODEL_ID='kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers'
MODEL_REVISION='4c487e8b4aeeb9059e5a77905aa2ddb59ec28f81'
OUTPUT_LIMIT_BYTES=19_500_000_000  # Kaggle keeps at most 20 GB of notebook output


In [ ]:
import hashlib, json, os, shutil, time
from pathlib import Path
os.environ.update(HF_HUB_DISABLE_TELEMETRY='1', HF_HUB_DISABLE_XET='1')
from huggingface_hub import HfApi, snapshot_download

# 'text' is the Qwen2.5-VL encoder alone; 'media' is everything else in the Diffusers layout.
PARTS = {'media': lambda path: not path.startswith('text_encoder/'),
         'text': lambda path: path.startswith('text_encoder/')}
OUT = Path('/kaggle/working/k6-lite-diffusers-'+PART)
files = [f for f in HfApi().list_repo_tree(MODEL_ID, revision=MODEL_REVISION, recursive=True)
         if getattr(f, 'size', None) is not None and PARTS[PART](f.path)]
need = sum(f.size for f in files)
free = shutil.disk_usage('/kaggle/working').free
print(f'{PART}: {len(files)} files, {need/2**30:.2f} GiB; free in /kaggle/working {free/2**30:.1f} GiB', flush=True)
if need > OUTPUT_LIMIT_BYTES:
    raise RuntimeError('This part does not fit into the Kaggle output limit')
if free < need*1.05:
    raise OSError('Not enough free space in /kaggle/working for this part')


In [ ]:
start = time.perf_counter()
snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_dir=OUT, allow_patterns=[f.path for f in files])
download_seconds = time.perf_counter()-start
# snapshot_download keeps download metadata here; it is not part of the Diffusers layout.
shutil.rmtree(OUT/'.cache', ignore_errors=True)
bad = [f.path for f in files if not (OUT/f.path).is_file() or (OUT/f.path).stat().st_size != f.size]
if bad:
    raise RuntimeError(f'Missing or truncated files: {bad}')
# Timed-out downloads are resumed by range requests, so the size alone does not prove the content.
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()
hash_start = time.perf_counter()
mismatched = [f.path for f in files if getattr(f, 'lfs', None) and sha256(OUT/f.path) != f.lfs.sha256]
if mismatched:
    raise RuntimeError(f'SHA-256 differs from the Hub for: {mismatched}')
print(f'SHA-256 matches the Hub for {sum(1 for f in files if getattr(f, "lfs", None))} LFS files '
      f'({time.perf_counter()-hash_start:.0f} s)', flush=True)
manifest = dict(model_id=MODEL_ID, revision=MODEL_REVISION, part=PART, total_bytes=need,
    download_seconds=round(download_seconds, 1), files={f.path: f.size for f in files},
    lfs_sha256={f.path: f.lfs.sha256 for f in files if getattr(f, 'lfs', None)}, lfs_sha256_verified=True)
(OUT/'weights-manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')
print(f'Saved {len(files)} files, {need/2**30:.2f} GiB in {download_seconds:.0f} s to {OUT}')
